In [1]:
import numpy as np
import pandas as pd

In [2]:
column_names = [
    "user_id",
    "movie_id",
    "rating",
    "timestamp"
]

train = pd.read_csv(
    "data/u1.base",
    sep="\t",
    names=column_names
)

test = pd.read_csv(
    "data/u1.test",
    sep="\t",
    names=column_names
)

In [3]:
users = pd.read_csv(
    "data/u.user",
    sep="|",
    names=[
        "user_id",
        "age",
        "gender",
        "occupation",
        "zip_code"
    ],
    encoding="latin-1"
)

In [4]:
genre_columns = [
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

movie_columns = [
    "movie_id",
    "title",
    "release_date",
    "video_release",
    "imdb_url"
] + genre_columns

movies = pd.read_csv(
    "data/u.item",
    sep="|",
    names=movie_columns,
    encoding="latin-1"
)

In [5]:
train_df = (
    train
    .merge(users, on="user_id")
    .merge(movies, on="movie_id")
)

In [6]:
bins = [0,18,25,35,45,50,60,100]

labels = [
    "0-18",
    "19-25",
    "26-35",
    "36-45",
    "46-50",
    "51-60",
    "60+"
]

train_df["age_group"] = pd.cut(
    train_df["age"],
    bins=bins,
    labels=labels
)

users["age_group"] = pd.cut(
    users["age"],
    bins=bins,
    labels=labels
)

In [7]:
feature_to_idx = {}

feature_counter = 0

genre_columns = [
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

for _, row in train_df.iterrows():

    features = [
        f"user_{row['user_id']}",
        f"movie_{row['movie_id']}",
        f"gender_{row['gender']}",
        f"occupation_{row['occupation']}",
        f"age_{row['age_group']}"
    ]

    # Add genres
    for genre in genre_columns:

        if row[genre] == 1:
            features.append(
                f"genre_{genre}"
            )

    # Build feature dictionary
    for feature in features:

        if feature not in feature_to_idx:

            feature_to_idx[feature] = feature_counter
            feature_counter += 1

In [8]:
print("Total Features:", len(feature_to_idx))

Total Features: 2642


In [9]:
training_samples = []

for _, row in train_df.iterrows():

    feature_indices = [

        feature_to_idx[f"user_{row['user_id']}"],

        feature_to_idx[f"movie_{row['movie_id']}"],

        feature_to_idx[f"gender_{row['gender']}"],

        feature_to_idx[f"occupation_{row['occupation']}"],

        feature_to_idx[f"age_{row['age_group']}"]

    ]

    # Genres
    for genre in genre_columns:

        if row[genre] == 1:

            feature_indices.append(
                feature_to_idx[
                    f"genre_{genre}"
                ]
            )

    training_samples.append(
        (
            feature_indices,
            row["rating"]
        )
    )

In [10]:
user_features = {}

for _, row in users.iterrows():

    age_group = pd.cut(
        [row["age"]],
        bins=[0,18,25,35,45,50,60,100],
        labels=[
            "0-18",
            "19-25",
            "26-35",
            "36-45",
            "46-50",
            "51-60",
            "60+"
        ]
    )[0]

    user_key = f"user_{row['user_id']}"

    # User never appeared in training
    if user_key not in feature_to_idx:
        continue

    user_features[row["user_id"]] = [

        feature_to_idx[user_key],

        feature_to_idx[f"gender_{row['gender']}"],

        feature_to_idx[f"occupation_{row['occupation']}"],

        feature_to_idx[f"age_{age_group}"]

    ]

In [11]:
movie_features = {}

genre_columns = movies.columns[5:]

for _, row in movies.iterrows():

    movie_key = f"movie_{row['movie_id']}"

    # Movie never appeared in training
    if movie_key not in feature_to_idx:
        continue

    features = [
        feature_to_idx[movie_key]
    ]

    for genre in genre_columns:

        genre_key = f"genre_{genre}"

        # Safety check (optional)
        if row[genre] == 1 and genre_key in feature_to_idx:
            features.append(
                feature_to_idx[genre_key]
            )

    movie_features[row["movie_id"]] = features

In [12]:
class FactorizationMachine:

    def __init__(
        self,
        num_features,
        user_features,
        movie_features,
        k=20,
        learning_rate=0.01,
        reg=0.02
    ):

        self.num_features = num_features
        
        self.user_features = user_features
        self.movie_features = movie_features
        
        self.k = k

        self.learning_rate = learning_rate
        self.reg = reg

        # Global Bias
        self.w0 = 0.0

        # Linear Weights
        self.w = np.zeros(num_features)

        # Feature Embeddings
        self.V = np.random.normal(
            0,
            0.01,
            (num_features, k)
        )
    
    
    def predict(
    self,
    feature_indices
    ):
        prediction = self.w0
    
        # Linear Part
        for feature in feature_indices:
        
            prediction += self.w[
                feature
            ]
    
        # Interaction Part
        V = self.V[np.array(feature_indices)]

        sum1 = np.sum(V, axis=0)

        sum2 = np.sum(V * V, axis=0)

        interaction = 0.5 * np.sum(
            sum1**2 - sum2
        )

        prediction += interaction
    
        return float(np.clip(prediction, 1, 5))
    

    def sgd_step(
        self,
        feature_indices,
        rating
    ):

        prediction = self.predict(
            feature_indices
        )

        error = rating - prediction

        # Global bias
        self.w0 += (
            self.learning_rate * error
        )

        # Linear weights
        for feature in feature_indices:

            self.w[feature] += (
                self.learning_rate
                * (
                    error
                    -
                    self.reg
                    * self.w[feature]
                )
            )

        # Sum of embeddings
        sum_v = np.zeros(self.k)

        for feature in feature_indices:

            sum_v += self.V[feature]

        # Update embeddings
        for feature in feature_indices:

            gradient = (
                sum_v
                -
                self.V[feature]
            )

            self.V[feature] += (

                self.learning_rate
                * (
                    error * gradient

                    -

                    self.reg
                    * self.V[feature]
                )

            )

        return error

    def fit(
        self,
        training_samples,
        epochs=20
    ):
    
        print("Training Factorization Machine...")
    
        for epoch in range(epochs):
        
            np.random.shuffle(
                training_samples
            )
    
            squared_errors = []
    
            for feature_indices, rating in training_samples:
            
                error = self.sgd_step(
                    feature_indices,
                    rating
                )
    
                squared_errors.append(
                    error ** 2
                )
    
            rmse = np.sqrt(
                np.mean(
                    squared_errors
                )
            )
    
            print(
                f"Epoch {epoch+1:2d}/{epochs}"
                f" | RMSE = {rmse:.4f}"
            )    

    def build_feature_vector(
        self,
        user_id,
        movie_id
    ):

        return [*self.user_features[user_id],*self.movie_features[movie_id]]


    def predict_rating(
        self,
        user_id,
        movie_id
    ):

        feature_indices = self.build_feature_vector(
            user_id,
            movie_id
        )

        return self.predict(
            feature_indices
        )


    def evaluate(
        self,
        test_df
    ):

        predictions = []

        squared_errors = []

        absolute_errors = []

        for row in test_df.itertuples(index=False):

            if (
                row.user_id not in self.user_features
                or
                row.movie_id not in self.movie_features
            ):
                continue

            predicted = self.predict_rating(
                row.user_id,
                row.movie_id
            )
            if np.isnan(predicted):
                continue

            error = abs(
                row.rating - predicted
            )

            predictions.append(
                (
                    row.user_id,
                    row.movie_id,
                    row.rating,
                    predicted,
                    error
                )
            )

            absolute_errors.append(error)
            squared_errors.append(error**2)

        prediction_df = pd.DataFrame(
            predictions,
            columns=[
                "user_id",
                "movie_id",
                "actual",
                "predicted",
                "error"
            ]
        )

        mae = np.mean(
            absolute_errors
        )

        rmse = np.sqrt(
            np.mean(
                squared_errors
            )
        )

        return prediction_df, rmse, mae


    def recommend(
        self,
        user_id,
        train_df,
        top_n=10
    ):

        watched = set(

            train_df.loc[
                train_df.user_id == user_id,
                "movie_id"
            ]

        )

        recommendations = []

        for movie_id in self.movie_features:

            if movie_id in watched:
                continue

            prediction = self.predict_rating(
                user_id,
                movie_id
            )

            recommendations.append(
                (
                    movie_id,
                    prediction
                )
            )

        recommendations.sort(
            key=lambda x: x[1],
            reverse=True
        )

        return pd.DataFrame(
            recommendations[:top_n],
            columns=[
                "movie_id",
                "predicted_rating"
            ]
        )

In [13]:
fm = FactorizationMachine(
    num_features=len(feature_to_idx),
    user_features=user_features,
    movie_features=movie_features,
    k=20,
    learning_rate=0.01,
    reg=0.02
)

fm.fit(
    training_samples,
    epochs=20
)

Training Factorization Machine...
Epoch  1/20 | RMSE = 1.0163
Epoch  2/20 | RMSE = 0.9588
Epoch  3/20 | RMSE = 0.9415
Epoch  4/20 | RMSE = 0.9289
Epoch  5/20 | RMSE = 0.9176
Epoch  6/20 | RMSE = 0.9078
Epoch  7/20 | RMSE = 0.8985
Epoch  8/20 | RMSE = 0.8888
Epoch  9/20 | RMSE = 0.8792
Epoch 10/20 | RMSE = 0.8701
Epoch 11/20 | RMSE = 0.8614
Epoch 12/20 | RMSE = 0.8532
Epoch 13/20 | RMSE = 0.8453
Epoch 14/20 | RMSE = 0.8368
Epoch 15/20 | RMSE = 0.8284
Epoch 16/20 | RMSE = 0.8196
Epoch 17/20 | RMSE = 0.8108
Epoch 18/20 | RMSE = 0.8038
Epoch 19/20 | RMSE = 0.7959
Epoch 20/20 | RMSE = 0.7869


In [14]:
fv = fm.build_feature_vector(1, 1)

print(type(fv))
print(fv)

<class 'list'>
[0, 2, 3, 4, 1, 5, 6, 7]


In [15]:
prediction_df, rmse, mae = fm.evaluate(
    test
)

print("MAE :", mae)
print("RMSE:", rmse)

prediction_df.head()

MAE : 0.7291162409162197
RMSE: 0.9355365999585787


,user_id,movie_id,actual,predicted,error
0,1,6,5,3.797755,1.202245
1,1,10,3,3.951071,0.951071
2,1,12,5,4.146900,0.853100
3,1,14,5,4.751619,0.248381
4,1,17,3,2.454162,0.545838


In [16]:
df=fm.recommend(
    user_id=432,
    train_df=train,
    top_n=10
)

In [17]:
df.merge(
    movies[["movie_id", "title"]],
    on="movie_id"
).sort_values(
    by="predicted_rating",
    ascending=False
)

,movie_id,predicted_rating,title
0,483,4.973961,Casablanca (1942)
1,172,4.899368,"Empire Strikes Back, The (1980)"
2,169,4.881676,"Wrong Trousers, The (1993)"
3,50,4.881292,Star Wars (1977)
4,173,4.763239,"Princess Bride, The (1987)"
5,408,4.711472,"Close Shave, A (1995)"
6,641,4.687911,Paths of Glory (1957)
7,12,4.666901,"Usual Suspects, The (1995)"
8,515,4.659762,"Boot, Das (1981)"
9,114,4.643375,Wallace & Gromit: The Best of Aardman Animatio...


In [18]:
def build_new_user_features(new_user):

    return [

        feature_to_idx[
            f"gender_{new_user['gender']}"
        ],

        feature_to_idx[
            f"occupation_{new_user['occupation']}"
        ],

        feature_to_idx[
            f"age_{new_user['age_group']}"
        ]

    ]

In [19]:
new_user = {
    "gender": "F",
    "occupation": "artist",
    "age_group": "26-35"
}

In [20]:
recommendations = []

new_features = build_new_user_features(
    new_user
)

for movie_id, movie_feat in movie_features.items():

    feature_vector = (
        new_features
        +
        movie_feat
    )

    prediction = fm.predict(
        feature_vector
    )

    recommendations.append(
        (
            movie_id,
            prediction
        )
    )
recommendations.sort(
    key=lambda x:x[1],
    reverse=True
)

In [21]:
recommendations[:10]

[(50, 5.0),
 (173, 4.909775760908207),
 (172, 4.782021824627328),
 (169, 4.768671385275851),
 (483, 4.619152768027286),
 (408, 4.606228278130731),
 (269, 4.532215231500481),
 (181, 4.444714155168738),
 (114, 4.412805137142731),
 (313, 4.360051146000089)]

In [22]:
cold_df = pd.DataFrame(
    recommendations[:10],
    columns=[
        "movie_id",
        "predicted_rating"
    ]
)

cold_df = cold_df.merge(
    movies[
        ["movie_id","title"]
    ],
    on="movie_id"
)

cold_df

,movie_id,predicted_rating,title
0,50,5.000000,Star Wars (1977)
1,173,4.909776,"Princess Bride, The (1987)"
2,172,4.782022,"Empire Strikes Back, The (1980)"
3,169,4.768671,"Wrong Trousers, The (1993)"
4,483,4.619153,Casablanca (1942)
5,408,4.606228,"Close Shave, A (1995)"
6,269,4.532215,"Full Monty, The (1997)"
7,181,4.444714,Return of the Jedi (1983)
8,114,4.412805,Wallace & Gromit: The Best of Aardman Animatio...
9,313,4.360051,Titanic (1997)
